In [1]:
import pandas as pd
import numpy as np

# 1. Load the clean baseline data
df = pd.read_csv('processed_data_v2.csv')

# 2. Ensure date and numeric columns are properly formatted
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['num_pages'] = pd.to_numeric(df['num_pages'], errors='coerce').fillna(0)

rating_col = 'rating' if 'rating' in df.columns else 'average_rating'
df['user_eval_rating'] = pd.to_numeric(df[rating_col], errors='coerce').fillna(0)

# Resolve author column dynamically ('author_id' or 'author_ids')
author_col = next((c for c in ['author_id', 'author_ids'] if c in df.columns), None)
if not author_col:
    author_col = 'author_id'
    df[author_col] = 'unknown'

df['genres'] = df['genres'].fillna('Unknown')
df[author_col] = df[author_col].fillna('unknown')

# 3. Sort chronologically so future books cannot leak into past metrics
df = df.sort_values(by=['user_id', 'date_added']).reset_index(drop=True)

# =======================================================
# A. Chronological Overall Conversion Rate
# =======================================================
df['prior_user_books'] = df.groupby('user_id').cumcount()
df['prior_user_conv'] = df.groupby('user_id')['converted_within_target'].cumsum() - df['converted_within_target']
df['overall_shelf_to_read_conversion'] = np.where(
    df['prior_user_books'] > 0, 
    df['prior_user_conv'] / df['prior_user_books'], 
    0.0
)

# =======================================================
# B. Chronological Book-Length Conversion Rates
# =======================================================
df['is_short'] = ((df['num_pages'] > 0) & (df['num_pages'] < 200)).astype(int)
df['is_medium'] = ((df['num_pages'] >= 200) & (df['num_pages'] <= 400)).astype(int)
df['is_long'] = (df['num_pages'] > 400).astype(int)

for cat in ['short', 'medium', 'long']:
    cat_flag = f'is_{cat}'
    cat_conv = f'{cat}_conv_flag'
    df[cat_conv] = df['converted_within_target'] * df[cat_flag]
    
    prior_cat_books = df.groupby('user_id')[cat_flag].cumsum() - df[cat_flag]
    prior_cat_conv = df.groupby('user_id')[cat_conv].cumsum() - df[cat_conv]
    
    df[f'{cat}_conversion_rate'] = np.where(
        prior_cat_books > 0, 
        prior_cat_conv / prior_cat_books, 
        0.0
    )

# =======================================================
# C. Chronological Genre Conversion & Rating Preference
# =======================================================
genre_grp = df.groupby(['user_id', 'genres'])
prior_genre_books = genre_grp.cumcount()
prior_genre_conv = genre_grp['converted_within_target'].cumsum() - df['converted_within_target']
prior_genre_rating = genre_grp['user_eval_rating'].cumsum() - df['user_eval_rating']

df['genre_conversion_rate'] = np.where(
    prior_genre_books > 0, 
    prior_genre_conv / prior_genre_books, 
    0.0
)
df['genre_preference'] = np.where(
    prior_genre_books > 0, 
    (prior_genre_rating / prior_genre_books) / 5.0, 
    0.0
)

# =======================================================
# D. Chronological Author Conversion Rate
# =======================================================
author_grp = df.groupby(['user_id', author_col])
prior_author_books = author_grp.cumcount()
prior_author_conv = author_grp['converted_within_target'].cumsum() - df['converted_within_target']

df['author_conversion_rate'] = np.where(
    prior_author_books > 0, 
    prior_author_conv / prior_author_books, 
    0.0
)

# =======================================================
# E. Select Feature Matrix & Export
# =======================================================
feature_cols = [
    'overall_shelf_to_read_conversion', 'short_conversion_rate', 
    'medium_conversion_rate', 'long_conversion_rate',
    'genre_conversion_rate', 'genre_preference', 'author_conversion_rate'
]

master_cols = ['user_id', 'book_id', 'converted_within_target'] + feature_cols
master_matrix_v2 = df[master_cols].copy()

output_matrix_file = 'book_recommender_master_feature_matrix_v2.csv'
master_matrix_v2.to_csv(output_matrix_file, index=False)

print("Chronological Master Feature Matrix Created Successfully!")
print(f"Total training rows: {len(master_matrix_v2)}")
print(f"File saved to: '{output_matrix_file}'\n")
display(master_matrix_v2.head(10))

Chronological Master Feature Matrix Created Successfully!
Total training rows: 41942
File saved to: 'looknew_master_feature_matrix_v2.csv'



,user_id,book_id,converted_within_target,overall_shelf_to_read_conversion,short_conversion_rate,medium_conversion_rate,long_conversion_rate,genre_conversion_rate,genre_preference,author_conversion_rate
0,0151a92d6ef1e6b28bb9c9b05777d79d,5107,0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
1,0151a92d6ef1e6b28bb9c9b05777d79d,2657,0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
2,0151a92d6ef1e6b28bb9c9b05777d79d,320,0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
3,0151a92d6ef1e6b28bb9c9b05777d79d,7437,0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
4,0151a92d6ef1e6b28bb9c9b05777d79d,412732,0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
5,0151a92d6ef1e6b28bb9c9b05777d79d,14942,0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
6,0151a92d6ef1e6b28bb9c9b05777d79d,56373,0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
7,0151a92d6ef1e6b28bb9c9b05777d79d,7588,0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
8,0151a92d6ef1e6b28bb9c9b05777d79d,360635,1,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000
9,0151a92d6ef1e6b28bb9c9b05777d79d,6241137,0,0.111111,0.0,0.0,0.0,0.0,0.0,0.111111
